<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/cmaes_acceleration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMA-ES 优化加速指南

在实际应用中，一个非常常见的问题是：**已经用上 CMA-ES，但整个优化过程仍然太慢，应该怎样加速？**

本 Notebook 面向已经在工程或研究中使用 CMA-ES 的读者，重点讨论在**不改变原问题定义**的前提下，如何从优化执行方式降低总时间。如果还不熟悉 CMA-ES 的基本使用方式，建议先阅读 `cmaes_practical_guide.ipynb`。

## 使用的 CMA-ES 实现

原教程使用作者公开的 DD-CMA-ES 实现，它在标准 CMA-ES 基础上加入 diagonal acceleration，并可配合矩形约束处理、周期变量和重启策略。

论文：Y. Akimoto and N. Hansen, *Diagonal Acceleration for Covariance Matrix Adaptation Evolution Strategies*, Evolutionary Computation 28(3), 2020.

参考代码：https://gist.github.com/youheiakimoto/1180b67b5a0b1265c204cba991fa8518

本仓库的 `6_advanced_adaptation_mechanisms.ipynb` 已经给出中文化的 dd-CMA 核心实现与算法解释，本 Notebook 不再重复复制整套类定义。

## 示例问题：Rastrigin

Rastrigin 函数定义为
$$f(x)=\sum_{i=1}^{d}x_i^2+10(1-\cos(2\pi x_i)).$$
它具有大量局部最优，但宏观上仍呈大谷结构，全局最优为 $x^*=(0,\dots,0)$。这类问题通常需要结合不同初始分布和种群规模的多次重启，因此特别适合讨论“总执行时间”而不是单次迭代时间。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rastrigin(x):
    x = np.asarray(x)
    return np.sum(x**2 + 10.0 * (1.0 - np.cos(2.0 * np.pi * x)), axis=-1)

# 候选解通常可以整批传入目标函数，从而方便后续做并行或向量化评估
X = np.random.randn(32, 10)
values = rastrigin(X)
values[:5]

# CMA-ES 总执行时间的分解

设一次候选解的目标函数评估耗时为 $\alpha$，CMA-ES 每次迭代除目标函数评估之外的内部开销为 $\beta$，每代种群规模为 $\lambda$。如果候选解串行评估，那么每次迭代耗时约为
$$\alpha\lambda+\beta.$$
若一次收敛需要 $T$ 次迭代，则总时间约为
$$T(\alpha\lambda+\beta).$$

多峰问题通常还需要重启。设第 $k$ 次运行的种群规模为 $\lambda_k$，需要 $T_k$ 次迭代，总共执行 $K$ 次运行，则总时间约为
$$\sum_{k=1}^{K}T_k(\alpha\lambda_k+\beta).$$

因此所谓“加速 CMA-ES”至少有三种不同含义：

1. 降低单次目标函数评估成本 $\alpha$；
2. 降低 CMA-ES 自身内部成本 $\beta$；
3. 减少达到目标解所需的函数评估次数，即降低 $T_k\lambda_k$。

实践中应先判断自己的瓶颈属于哪一种。对仿真驱动优化，通常 $\alpha\gg\beta$，因此首先优化 CMA-ES 的矩阵运算往往并不是最有效的方向。

# 第一优先级：候选解并行评估

最实用、也最值得优先考虑的加速方式，是并行评估同一代生成的 $\lambda$ 个候选解。它们在 CMA-ES 的 `ask` 与 `tell` 之间互不依赖，因此天然可以并行。

如果有至少 $\lambda$ 个并行 worker，单代时间由
$$\alpha\lambda+\beta$$
近似降低为
$$\alpha+\beta.$$

如果只能并行执行 $m<\lambda$ 个任务，则单代时间约为
$$\alpha\left\lceil\frac{\lambda}{m}\right\rceil+\beta.$$

当 $\alpha$ 很大时，这通常带来接近 $m$ 倍的墙钟时间加速。注意这里减少的是**wall-clock time**，函数评估总次数并没有减少。

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def expensive_objective(x):
    # 实际应用中可替换为仿真、外部程序或远程评估
    return float(np.sum(np.asarray(x)**2))

candidates = np.random.randn(16, 20)
with ThreadPoolExecutor(max_workers=8) as pool:
    fitness = np.asarray(list(pool.map(expensive_objective, candidates)))
fitness[:5]

## 并行化时需要区分三种情况

**CPU 纯 Python 计算**：如果目标函数受 Python GIL 限制，应优先用多进程而不是线程。

**外部仿真 / I/O / RPC**：线程、进程、作业调度系统都可以；关键是让一代候选解批量提交，而不是逐个阻塞等待。

**GPU / 向量化模型**：通常不应启动 $\lambda$ 个独立 Python 任务，而应把候选解堆成 batch，一次送入 GPU。此时评估成本更接近一次大 batch 推理，而不是 $\lambda$ 次独立推理。

# 种群规模不是越小越快

从单代时间公式看，减小 $\lambda$ 好像总能加速，但这并不成立，因为 $T$ 会随 $\lambda$ 改变。较大的种群能够提供更可靠的排序与协方差估计，在多峰问题上还可能提高跳出局部最优并找到更好盆地的概率。

因此真正应该比较的是**达到同一目标精度所需的总函数评估数与墙钟时间**，而不是只比较每代耗时。对于多峰问题，IPOP-CMA-ES 逐次增大种群规模的原因正是：小种群适合廉价局部搜索，大种群则逐渐提升全局探索能力。

# 什么时候值得优化 CMA-ES 内部开销 $\beta$

如果目标函数非常便宜、维数很高，此时 $\beta$ 才可能成为主要成本。标准完整 CMA-ES 需要维护和分解 $N\times N$ 协方差矩阵，时间和内存成本会随维数快速增加。可以考虑：

* **Separable-CMA-ES**：只维护对角方差，时间和空间近似线性，但失去变量相关性建模能力；
* **Diagonal Decoding / dd-CMA-ES**：快速学习坐标尺度，同时保留完整相关矩阵，兼顾可分与旋转问题；
* 降低特征值分解频率，而不是每代都完整分解；
* 如果问题结构已知近似可分，可以有意识地选择对角版本，而不是盲目使用完整矩阵。

这些机制的原理和实验见 `3_covariance_matrix_adaptation.ipynb`、`4_nonseparability.ipynb` 与 `6_advanced_adaptation_mechanisms.ipynb`。

# 比“算得更快”更重要：减少无效函数评估

如果一次仿真非常昂贵，真正的加速往往来自减少函数评估次数，而不是优化 CMA-ES 本身。建议依次检查：

* 初始均值是否位于合理搜索区域；
* 初始尺度是否和每个变量的物理范围相匹配；
* 是否存在明显的变量尺度差异，需要使用 per-coordinate scaling / diagonal decoding；
* 约束处理是否导致大量候选解被裁剪到同一点或落入无效区域；
* 是否应该使用重启，而不是让已经停滞的一次运行继续消耗预算；
* 多峰问题是否需要逐步增大种群，而不是一开始就使用极大种群；
* 是否能够使用低保真模型、代理模型或多保真策略减少昂贵仿真的调用。

这些措施改变的是 $T_k$ 或有效评估比例，通常比只优化矩阵运算带来的收益更大。

# 实践中的加速顺序

对于一个正在运行的 CMA-ES 工程，可以按下面顺序排查：

1. **测量 $\alpha$ 与 $\beta$**：先确认时间究竟花在哪里。
2. **批量 / 并行候选解评估**：仿真昂贵时通常收益最大。
3. **检查初始尺度和约束处理**：避免大量无效评估。
4. **使用合理终止条件与重启**：及时停止已经停滞的搜索。
5. **比较总函数评估预算，而不是单代速度**。
6. 只有当目标函数便宜且维数很高时，再重点优化 CMA-ES 的 $\beta$，例如使用 Separable-CMA 或 dd-CMA。

这个顺序的核心思想是：**CMA-ES 的主要成本通常不是“生成候选解”，而是“评价候选解”。**